# YOLOv8 IndoorObjects-Person + HomeObjects-3K
Colab-friendly T4 training notebook.

This version removes the broken Ultralytics HUB dataset-download flow and replaces it with a local conversion pipeline for `indoorobjects-person` from the provided NDJSON export, then combines it with the official HomeObjects-3K dataset.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # choose indoorobjects-person.ndjson from your computer
NDJSON_PATH = Path(next(iter(uploaded.keys())))

print("Using:", NDJSON_PATH)
print("Exists:", NDJSON_PATH.exists())

Saving indoorobjects-person.ndjson to indoorobjects-person (1).ndjson
Using: indoorobjects-person (1).ndjson
Exists: True


In [ ]:
# 2) If you again hit the exact "numpy.dtype size changed" error,
# run ONLY this one line, then restart runtime once:
!pip -q install "numpy<2"

In [ ]:
# Clean Colab runtime first (Runtime -> Factory reset runtime)

from pathlib import Path
import os, json, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import yaml
import torch
from tqdm.auto import tqdm

!pip install ultralytics

from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import DATASETS_DIR

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

HOME = Path("/content")
ROOT = Path("/content/datasets")
ROOT.mkdir(parents=True, exist_ok=True)
print("HOME:", HOME)
print("DATASETS ROOT:", ROOT)

GPU available: True
GPU: Tesla T4
HOME: /content
DATASETS ROOT: /content/datasets


## IndoorObjects-Person source
Upload `indoorobjects-person.ndjson` into Colab (or put it in Drive and point `NDJSON_PATH` to it).  
The file contains image URLs plus normalized YOLO boxes, so we can build a local YOLO dataset without any Ultralytics API key.

In [ ]:
# =========================================================
# 3) Build IndoorObjects-Person local YOLO dataset from NDJSON
# =========================================================
NDJSON_PATH = Path("/content/indoorobjects-person.ndjson")
INDOOR_ROOT = ROOT / "indoorobjects-person"
INDOOR_ROOT.mkdir(parents=True, exist_ok=True)

if not NDJSON_PATH.exists():
    raise FileNotFoundError(
        f"{NDJSON_PATH} not found.\n"
        "Upload the NDJSON export to /content or change NDJSON_PATH to your Drive path."
    )

def safe_name(name: str) -> str:
    return Path(name).name

def download_image(session: requests.Session, url: str, out_path: Path, timeout: int = 90) -> bool:
    if out_path.exists() and out_path.stat().st_size > 0:
        return True
    out_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        r = session.get(url, timeout=timeout, stream=True)
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 64):
                if chunk:
                    f.write(chunk)
        return out_path.exists() and out_path.stat().st_size > 0
    except Exception as e:
        print(f"Download failed: {out_path.name} -> {e}")
        return False

with open(NDJSON_PATH, "r", encoding="utf-8") as f:
    meta = json.loads(f.readline())
    indoor_class_names = [meta["class_names"][str(i)] for i in range(len(meta["class_names"]))]

    image_records = [json.loads(line) for line in f if line.strip()]
    image_records = [r for r in image_records if r.get("type") == "image"]

print(f"IndoorObjects-Person classes ({len(indoor_class_names)}): {indoor_class_names}")
print(f"Records: {len(image_records)}")

# Prepare folders
for split in ("train", "val"):
    (INDOOR_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (INDOOR_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Download images + write labels
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

def process_record(rec):
    split = rec.get("split", "train")
    if split not in ("train", "val"):
        split = "train"

    img_file = safe_name(rec["file"])
    img_path = INDOOR_ROOT / "images" / split / img_file
    label_path = INDOOR_ROOT / "labels" / split / f"{Path(img_file).stem}.txt"

    ok = download_image(session, rec["url"], img_path)
    if not ok:
        return False

    boxes = rec.get("annotations", {}).get("boxes", [])
    with open(label_path, "w", encoding="utf-8") as lf:
        for box in boxes:
            cls, xc, yc, w, h = box
            lf.write(f"{int(cls)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
    return True

ok_count = 0
with ThreadPoolExecutor(max_workers=12) as ex:
    futures = [ex.submit(process_record, rec) for rec in image_records]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Building IndoorObjects-Person"):
        if fut.result():
            ok_count += 1

print(f"Built {ok_count}/{len(image_records)} images")
print("Indoor dataset root:", INDOOR_ROOT)

IndoorObjects-Person classes (13): ['bed', 'sofa', 'chair', 'table', 'lamp', 'tv', 'laptop', 'wardrobe', 'window', 'door', 'potted plant', 'photo frame', 'person']
Records: 2986


Building IndoorObjects-Person:   0%|          | 0/2986 [00:00<?, ?it/s]

Built 2986/2986 images
Indoor dataset root: /content/datasets/indoorobjects-person


In [ ]:
# =========================================================
# 5) Create YAML for CUSTOM dataset only
# =========================================================
# Define CUSTOM_ROOT and custom_class_names using the processed IndoorObjects-Person dataset
CUSTOM_ROOT = INDOOR_ROOT
custom_class_names = indoor_class_names

CUSTOM_YAML = CUSTOM_ROOT / "data.yaml"

custom_cfg = {
    "path": str(CUSTOM_ROOT),
    "train": "images/train",
    "val": "images/val",
    "nc": len(custom_class_names),
    "names": custom_class_names,
}

with open(CUSTOM_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(custom_cfg, f, sort_keys=False)

print("Saved YAML to:", CUSTOM_YAML)
print(CUSTOM_YAML.read_text())

Saved YAML to: /content/datasets/indoorobjects-person/data.yaml
path: /content/datasets/indoorobjects-person
train: images/train
val: images/val
nc: 13
names:
- bed
- sofa
- chair
- table
- lamp
- tv
- laptop
- wardrobe
- window
- door
- potted plant
- photo frame
- person



In [ ]:
# =========================================================
# 6) Train YOLOv8 on T4
# =========================================================
# Use yolov8n.pt for the safest Colab/T4 setup. Change to yolov8s.pt if you want more capacity.
MODEL_WEIGHTS = "yolov8n.pt"
EPOCHS = 50
IMG_SIZE = 640
PROJECT = str(HOME / "runs" / "detect")
RUN_NAME = "indoor_home_combined"

model = YOLO(MODEL_WEIGHTS)

results = model.train(
    data=str(CUSTOM_YAML),
    epochs=50,
    imgsz=IMG_SIZE,
    device=0,
    batch=-1,
    workers=2,
    amp=True,
    cache=False,
    patience=20,
    close_mosaic=10,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)

print("Training finished.")

Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/indoorobjects-person/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=indoor_home_combined, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

ModuleNotFoundError: No module named 'numpy.char'

In [ ]:
# =========================================================
# 8) Validate, predict, export, download best model
# =========================================================
RUN_DIR = Path(PROJECT) / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"

if BEST_PT.exists():
    best_model = YOLO(str(BEST_PT))

    # Validation on custom dataset only
    val_metrics = best_model.val(data=str(CUSTOM_YAML), device=0)
    print(val_metrics)

    # Quick inference example
    test_image = HOME / "test.jpg"
    if not test_image.exists():
        !wget -q https://ultralytics.com/images/bus.jpg -O /content/test.jpg

    pred = best_model.predict(
        source=str(test_image),
        conf=0.25,
        save=True,
        imgsz=IMG_SIZE
    )
    print("Prediction saved under runs/detect/predict")

    # Export ONNX (optional)
    best_model.export(format="onnx", imgsz=IMG_SIZE, dynamic=True, simplify=True)

    # Download best weights
    from google.colab import files
    files.download(str(BEST_PT))
else:
    print(f"best.pt not found at {BEST_PT}")

Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,008,183 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2337.1±852.5 MB/s, size: 259.2 KB)
val: Scanning /content/datasets/indoorobjects-person/labels/val.cache... 443 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 443/443 92.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 3.0it/s 9.2s
                   all        443       3826      0.742      0.633      0.693      0.481
                   bed         22         22      0.732      0.682      0.692      0.518
                  sofa        285        397      0.851      0.778      0.856      0.648
                 chair        154        305      0.703      0.685      0.699      0.491
                 table        299        467      0.771      0.719      0.802      0.587
                  lamp      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>